In [1]:
!pip install sacrebleu rouge-score matplotlib torch transformers datasets pandas evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.3 MB/s eta 0:00:00


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
translation_metrics.py
======================
Shared evaluation metrics and training-curve plotting utilities.
Used by all three translator scripts (BanglaT5, BanglaBERT-Enc-Dec, mBART, mT5).

Metrics computed per checkpoint / epoch
----------------------------------------
  • BLEU      (sacrebleu)   — n-gram precision with brevity penalty
  • ROUGE-L   (rouge_score) — longest common subsequence recall
  • chrF      (sacrebleu)   — character n-gram F-score (robust for morphological langs)
  • TER       (sacrebleu)   — translation edit rate (lower = better)

Plot produced after training
-----------------------------
  • One multi-panel PNG per model with Train Loss, Eval Loss,
    BLEU, ROUGE-L, chrF, and TER curves over epochs.

Install requirements (once):
    pip install sacrebleu rouge-score matplotlib
"""

import os
import json
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe for Kaggle/Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import evaluate
from rouge_score import rouge_scorer as _rouge_scorer


# ─────────────────────────────────────────────────────────────────────────────
# Metric loaders (loaded once, reused)
# ─────────────────────────────────────────────────────────────────────────────

_bleu  = evaluate.load("sacrebleu")
_chrf  = evaluate.load("chrf")
_ter   = evaluate.load("ter")
_rouge = _rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)


def compute_all_metrics(decoded_preds: list[str],
                        decoded_labels: list[str]) -> dict[str, float]:
    """
    Compute BLEU, ROUGE-L, chrF, and TER for a batch of predictions.

    Args:
        decoded_preds  : list of plain-text predictions (already stripped)
        decoded_labels : list of plain-text references  (already stripped)

    Returns:
        dict with keys: bleu, rougeL, chrF, ter
    """
    refs_for_bleu = [[l] for l in decoded_labels]   # sacrebleu expects list-of-lists

    bleu_score = _bleu.compute(
        predictions=decoded_preds, references=refs_for_bleu
    )["score"]

    chrf_score = _chrf.compute(
        predictions=decoded_preds, references=refs_for_bleu,
        word_order=2                  # chrF++ (also counts word bigrams)
    )["score"]

    ter_score = _ter.compute(
        predictions=decoded_preds, references=refs_for_bleu,
        normalized=True
    )["score"]

    # ROUGE-L — average over the batch
    rl_scores = [
        _rouge.score(ref, pred)["rougeL"].fmeasure
        for pred, ref in zip(decoded_preds, decoded_labels)
    ]
    rougeL_score = float(np.mean(rl_scores)) * 100   # scale to 0-100 like BLEU

    return {
        "bleu":   round(bleu_score,   4),
        "rougeL": round(rougeL_score, 4),
        "chrF":   round(chrf_score,   4),
        "ter":    round(ter_score,    4),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Training-log parser
# ─────────────────────────────────────────────────────────────────────────────

def parse_trainer_log(log_history: list[dict]) -> dict[str, list]:
    """
    Extract per-epoch training/eval curves from HF Trainer's log_history.

    Returns a dict:
        epochs      : list of epoch numbers
        train_loss  : mean training loss per epoch
        eval_loss   : eval loss per epoch
        bleu        : eval BLEU per epoch
        rougeL      : eval ROUGE-L per epoch
        chrF        : eval chrF per epoch
        ter         : eval TER per epoch
    """
    train_steps: dict[int, list] = {}
    eval_rows:   dict[int, dict] = {}

    for entry in log_history:
        ep = entry.get("epoch")
        if ep is None:
            continue
        ep_int = math.ceil(ep)

        if "loss" in entry and "eval_loss" not in entry:
            train_steps.setdefault(ep_int, []).append(entry["loss"])
        elif "eval_loss" in entry:
            eval_rows[ep_int] = entry

    epochs     = sorted(eval_rows.keys())
    train_loss = [float(np.mean(train_steps.get(e, [float("nan")])))
 for e in epochs]
    eval_loss  = [eval_rows[e].get("eval_loss",  float("nan")) for e in epochs]
    bleu       = [eval_rows[e].get("eval_bleu",  float("nan")) for e in epochs]
    rougeL     = [eval_rows[e].get("eval_rougeL",float("nan")) for e in epochs]
    chrf       = [eval_rows[e].get("eval_chrF",  float("nan")) for e in epochs]
    ter        = [eval_rows[e].get("eval_ter",   float("nan")) for e in epochs]

    return dict(epochs=epochs, train_loss=train_loss, eval_loss=eval_loss,
                bleu=bleu, rougeL=rougeL, chrF=chrf, ter=ter)


# ─────────────────────────────────────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────────────────────────────────────

_PALETTE = {
    "train": "#2563EB",   # blue
    "eval":  "#DC2626",   # red
    "bleu":  "#16A34A",   # green
    "rougeL":"#9333EA",   # purple
    "chrF":  "#EA580C",   # orange
    "ter":   "#0891B2",   # cyan
}


def plot_training_curves(log_history: list[dict],
                         model_name:  str,
                         save_dir:    str = ".") -> str:
    """
    Parse HF Trainer log_history and save a 6-panel metrics plot.

    Args:
        log_history : trainer.state.log_history
        model_name  : e.g. "BanglaT5", "mBART", "mT5"
        save_dir    : directory to write the PNG into

    Returns:
        Path to the saved PNG file.
    """
    data = parse_trainer_log(log_history)
    epochs = data["epochs"]

    if not epochs:
        print("⚠️  No epoch data found in log_history — skipping plot.")
        return ""

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f"{model_name} — Training Curves", fontsize=16, fontweight="bold", y=1.01)

    panels = [
        ("Train Loss",  "Loss",    [(data["train_loss"], "Train",  _PALETTE["train"])]),
        ("Eval Loss",   "Loss",    [(data["eval_loss"],  "Eval",   _PALETTE["eval"])]),
        ("BLEU ↑",      "Score",   [(data["bleu"],       "BLEU",   _PALETTE["bleu"])]),
        ("ROUGE-L ↑",   "Score",   [(data["rougeL"],     "ROUGE-L",_PALETTE["rougeL"])]),
        ("chrF++ ↑",    "Score",   [(data["chrF"],       "chrF++", _PALETTE["chrF"])]),
        ("TER ↓",       "Score",   [(data["ter"],        "TER",    _PALETTE["ter"])])
    ]

    for ax, (title, ylabel, series) in zip(axes.flat, panels):
        for values, label, color in series:
            clean = [v if not (v is None or (isinstance(v, float) and math.isnan(v))) else None
                     for v in values]
            xs = [e for e, v in zip(epochs, clean) if v is not None]
            ys = [v for v in clean if v is not None]
            if xs:
                ax.plot(xs, ys, marker="o", markersize=5, linewidth=2,
                        color=color, label=label)
                # annotate last point
                ax.annotate(f"{ys[-1]:.2f}",
                            xy=(xs[-1], ys[-1]),
                            xytext=(4, 4), textcoords="offset points",
                            fontsize=8, color=color)

        ax.set_title(title, fontsize=11, fontweight="semibold")
        ax.set_xlabel("Epoch", fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.legend(fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, f"{model_name.replace(' ', '_')}_training_curves.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"📊 Training curves saved → {out_path}")
    return out_path


def save_metrics_json(metrics: dict, model_name: str, save_dir: str = ".") -> str:
    """Persist a final metrics dict to JSON for later comparison."""
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, f"{model_name.replace(' ', '_')}_metrics.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"💾 Metrics saved → {path}")
    return path

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
BanglaT5 Translation Pipeline  (Updated — multi-metric + plots)
================================================================
Metrics : BLEU, ROUGE-L, chrF++, TER
Plots   : 6-panel training curves saved as PNG after training
No wandb.
"""

import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

os.environ["WANDB_DISABLED"] = "true"

MODEL_NAME = "csebuetnlp/banglat5"


class SimpleTranslator:
    """
    BanglaT5-based Bangla ↔ Chakma translation pipeline.

    Usage:
        translator = SimpleTranslator('data.csv')
        translator.train()
        result = translator.translate("আমি খুশি।")
    """

    def __init__(self, csv_file, bangla_col="Bangla", chakma_col="Chakma",
                 bangla_to_chakma=True, epochs=10):
        self.csv_file   = csv_file
        self.bangla_col = bangla_col
        self.chakma_col = chakma_col
        self.direction  = bangla_to_chakma
        self.epochs     = epochs

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🚀 Using device: {self.device}")

        print("📥 Loading BanglaT5 model...")
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        self.model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        self.model.to(self.device)

        self._load_data()
        print("✅ Ready to train!")

    # ── Data ──────────────────────────────────────────────────────────────────

    def _load_data(self):
        print(f"📂 Loading data from {self.csv_file}...")
        df = pd.read_csv(self.csv_file)
        df = df[[self.bangla_col, self.chakma_col]].dropna()
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

        n = len(df)
        train_end, val_end = int(0.8 * n), int(0.9 * n)
        splits = {"train": df[:train_end], "validation": df[train_end:val_end], "test": df[val_end:]}
        print(f"📊 Train: {len(splits['train'])} | Val: {len(splits['validation'])} | Test: {len(splits['test'])}")

        self.dataset = DatasetDict({
            s: Dataset.from_dict({"bangla": f[self.bangla_col].tolist(),
                                  "chakma": f[self.chakma_col].tolist()})
            for s, f in splits.items()
        })
        self._tokenize()

    def _tokenize(self):
        print("🔠 Tokenizing...")

        def tokenize_fn(examples):
            src = examples["bangla"] if self.direction else examples["chakma"]
            tgt = examples["chakma"] if self.direction else examples["bangla"]
            model_inputs = self.tokenizer(src, max_length=128, truncation=True)
            labels       = self.tokenizer(tgt, max_length=128, truncation=True)
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

        self.tokenized = self.dataset.map(tokenize_fn, batched=True,
                                          remove_columns=["bangla", "chakma"])

    # ── Training ──────────────────────────────────────────────────────────────

    def train(self, save_dir: str = "./kaggle/working/saved_model"):
        print(f"\n🎓 Training for {self.epochs} epochs...")

        args = Seq2SeqTrainingArguments(
            output_dir=save_dir,
            num_train_epochs=self.epochs,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            learning_rate=3e-4,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            predict_with_generate=True,
            generation_max_length=128,
            generation_num_beams=5,
            logging_steps=50,
            save_total_limit=2,
            metric_for_best_model="bleu",
            greater_is_better=True,
            fp16=torch.cuda.is_available(),
            report_to="none",
        )

        collator = DataCollatorForSeq2Seq(self.tokenizer, model=self.model)

        def compute_metrics(eval_preds):
            preds, labels = eval_preds
            if isinstance(preds, tuple):
                preds = preds[0]
            preds = np.array(preds)
            if preds.dtype.kind == "f":
                preds = np.argmax(preds, axis=-1)
            preds  = np.clip(preds.astype(np.int64), 0, self.tokenizer.vocab_size - 1)
            labels = np.where(labels != -100, labels, self.tokenizer.pad_token_id)

            decoded_preds  = [p.strip() for p in self.tokenizer.batch_decode(preds,  skip_special_tokens=True)]
            decoded_labels = [l.strip() for l in self.tokenizer.batch_decode(labels, skip_special_tokens=True)]

            return compute_all_metrics(decoded_preds, decoded_labels)

        self.trainer = Seq2SeqTrainer(
            model=self.model, args=args,
            train_dataset=self.tokenized["train"],
            eval_dataset=self.tokenized["validation"],
            data_collator=collator,
            compute_metrics=compute_metrics,
        )

        self.trainer.train()

        # ── Test evaluation ───────────────────────────────────────────────────
        print("\n📊 Evaluating on test set...")
        test_results = self.trainer.evaluate(self.tokenized["test"])
        print(f"  BLEU   : {test_results.get('eval_bleu',   0):.2f}")
        print(f"  ROUGE-L: {test_results.get('eval_rougeL', 0):.2f}")
        print(f"  chrF++ : {test_results.get('eval_chrF',   0):.2f}")
        print(f"  TER    : {test_results.get('eval_ter',    0):.2f}")

        # ── Plots ─────────────────────────────────────────────────────────────
        plot_training_curves(
            self.trainer.state.log_history,
            model_name="BanglaT5",
            save_dir=save_dir,
        )
        save_metrics_json(test_results, model_name="BanglaT5", save_dir=save_dir)
        return test_results

    # ── Inference ─────────────────────────────────────────────────────────────

    def translate(self, text: str) -> str:
        self.model.eval()
        inputs = self.tokenizer(text, return_tensors="pt", max_length=128,
                                truncation=True).to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=128, num_beams=5)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def save(self, path: str = "./kaggle/working/saved_model"):
        print(f"💾 Saving to {path}...")
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print("✅ Saved!")


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("TRAINING BANGLA → CHAKMA  (BanglaT5)")
    print("=" * 60 + "\n")

    translator = SimpleTranslator(
        csv_file="/kaggle/input/datasets/arqambinalmas/thesis-dataset/Thesis_Dataset.csv",
        bangla_col="Bangla", chakma_col="Chakma",
        bangla_to_chakma=True, epochs=15,
    )
    translator.train(save_dir="./banglat5_results")

    print("\n" + "=" * 60)
    print("TESTING TRANSLATIONS")
    print("=" * 60 + "\n")

    for sentence in ["আমি খুশি।", "তুমি কেমন আছো?", "আজ আবহাওয়া ভালো।"]:
        print(f"📝 Bangla: {sentence}")
        print(f"   Chakma: {translator.translate(sentence)}\n")

    translator.save("./banglat5_Bangla2Chakma_model")
    print("\n✅ Done!")

TRAINING BANGLA → CHAKMA  (BanglaT5)

🚀 Using device: cuda
📥 Loading BanglaT5 model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 1.11MB            

spiece.model: downloading bytes:           |  0.00B            

...

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Bleu,Rougel,Chrf,Ter
2.488026,2.171191,15,23.259600,0.109100,43.893900,64.654200


  BLEU   : 23.26
  ROUGE-L: 0.11
  chrF++ : 43.89
  TER    : 64.65
📊 Training curves saved → ./banglat5_results/BanglaT5_training_curves.png
💾 Metrics saved → ./banglat5_results/BanglaT5_metrics.json

TESTING TRANSLATIONS

📝 Bangla: আমি খুশি।
   Chakma: মুই খুজি।

📝 Bangla: তুমি কেমন আছো?
   Chakma: তুই খেন্জান্ আগজ্?

📝 Bangla: আজ আবহাওয়া ভালো।
   Chakma: ইচ্ছি দেবা আন দোল।

💾 Saving to ./banglat5_Bangla2Chakma_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved!

✅ Done!
